### VT-NIRS: Virtual Twin for Non-Invasive Respiratory Support

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)   


### Library Loading

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

import torch

from models.vt_nirs import VTNIRSModel
from models.baselines import CustomTLearner, CustomCausalForest, run_baselines
from utils.losses import CensoringAwareAdversarialLoss
from utils.metrics import (
    compute_all_metrics, c_for_benefit, plot_model_comparison_bars,
    plot_ite_distribution, plot_training_curves,
    plot_decomposed_ite_scatter, plot_subgroup_ite_trends
)
from utils.loader import COVARIATES_23, create_dataloaders, NIRSTwinDataset
from utils.extraction import (
    init_client, run_mimic_extraction, propensity_score_match,
    normalize_and_mask, standardize_features,
    extract_eicu_cohort, assign_eicu_treatment,
    extract_eicu_covariates, compute_eicu_vfd28,
    extract_eicu_temporal, build_eicu_temporal_sequences,
    propensity_score_match_baseline,
    FEATURE_COLS, TEMPORAL_FEATURE_COLS, N_TEMPORAL_COVARIATES,
)
from training.train import train_stage1, train_stage2, evaluate, DEFAULT_CONFIG

warnings.filterwarnings('ignore')
print('All imports successful.')
print(f'PyTorch: {torch.__version__} | Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
BILLING_PROJECT = "YOUR_GCP_PROJECT_ID"  # <-- Set your Google Cloud billing project ID here

import utils.extraction as _ext
_ext.BILLING_PROJECT = BILLING_PROJECT
print(f'Billing project set to: {BILLING_PROJECT}')

In [ ]:
RANDOM_STATE   = 42
SEQUENCE_LEN   = 48
N_COVARIATES   = 23
D_MODEL        = 128
N_HEADS        = 4
N_LAYERS       = 4
D_FF           = 256
NOISE_DIM      = 8
HIDDEN_DIM     = 128
DROPOUT        = 0.1
BATCH_SIZE     = 128
EPOCHS_STAGE1  = 100
EPOCHS_STAGE2  = 50
LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 20
LAMBDA_ADV     = 1.0
LAMBDA_SURV    = 1.0
LAMBDA_VFD     = 0.5
LAMBDA_CONSIST = 0.5
LAMBDA_GATE    = 0.1
LAMBDA_IPM     = 1.0
LAMBDA_DR      = 2.0
LAMBDA_GP      = 10.0

OUTPUT_DIR = os.path.join(os.getcwd(), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'Config loaded. Device: {DEVICE}')


### MIMIC-IV Data Extraction

In [ ]:
print('Running MIMIC-IV BigQuery extraction...')
client = init_client()
data = run_mimic_extraction(client, seq_len=SEQUENCE_LEN)

X_all     = data['X']
W_all     = data['W']
VFD_all   = data['VFD']
D_all     = data['D']
valid_ids = data['valid_ids']
df_cohort = data['df_cohort']
df_vfd    = data['df_vfd']
df_cov    = data['df_cov']

X_baseline = df_cov[FEATURE_COLS].values
print(f'MIMIC-IV: X={X_all.shape}, N={len(df_cohort):,} patients')

### Propensity Score Matching & DataLoaders

In [ ]:
psm_idx, ps_model, ps = propensity_score_match(
    X_all, W_all, caliper_scale=0.2, random_state=RANDOM_STATE)

X_psm = X_all[psm_idx]
W_psm = W_all[psm_idx]
VFD_psm = VFD_all[psm_idx]
D_psm = D_all[psm_idx]

print(f'NIRS mean VFD-28: {VFD_psm[W_psm==1].mean():.2f}')
print(f'IMV  mean VFD-28: {VFD_psm[W_psm==0].mean():.2f}')

In [ ]:
X_scaled, pad_masks, ts_scaler = normalize_and_mask(X_psm, N_COVARIATES)

train_loader, val_loader, test_loader = create_dataloaders(
    sequences=X_scaled, treatments=W_psm,
    vfd_observed=VFD_psm, delta=D_psm,
    pad_masks=pad_masks, batch_size=BATCH_SIZE,
    val_fraction=0.15, test_fraction=0.15,
    random_state=RANDOM_STATE)

sample_batch = next(iter(train_loader))
for k, v in sample_batch.items():
    print(f'  {k}: {v.shape}')

### Model Initialization & Training

In [ ]:
model = VTNIRSModel(
    n_covariates=N_COVARIATES, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
).to(DEVICE)

loss_fn = CensoringAwareAdversarialLoss(
    lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV,
    lambda_vfd=LAMBDA_VFD, lambda_consist=LAMBDA_CONSIST,
    lambda_gate=LAMBDA_GATE,
    lambda_ipm=LAMBDA_IPM,
    lambda_dr=LAMBDA_DR,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'  Propensity head params: {sum(p.numel() for p in model.propensity_head.parameters()):,}')
print(f'  Discriminator params: {sum(p.numel() for p in model.discriminator.parameters()):,}')

model.eval()
with torch.no_grad():
    pred_test, enc_out = model.forward_predictor(
        sample_batch['x'].to(DEVICE), sample_batch['pad_mask'].to(DEVICE))
print(f'Forward pass OK. ITE shape: {pred_test["ite"].shape}')
print(f'Propensity logits shape: {enc_out[5].shape}')

In [ ]:
config = DEFAULT_CONFIG.copy()
config.update({
    'n_covariates': N_COVARIATES, 'd_model': D_MODEL, 'n_heads': N_HEADS,
    'n_layers': N_LAYERS, 'd_ff': D_FF, 'noise_dim': NOISE_DIM,
    'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT,
    'epochs_stage1': EPOCHS_STAGE1, 'lr_generator': LEARNING_RATE,
    'lr_discriminator': LEARNING_RATE, 'lr_encoder': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'batch_size': BATCH_SIZE,
    'device': str(DEVICE), 'save_dir': OUTPUT_DIR, 'patience': PATIENCE,
    'lambda_ipm': LAMBDA_IPM, 'lambda_dr': LAMBDA_DR, 'lambda_gp': LAMBDA_GP,
})

print(f'Stage 1: {EPOCHS_STAGE1} epochs, LR={LEARNING_RATE}, patience={PATIENCE}')
t0 = time.time()
model, log_stage1 = train_stage1(model, train_loader, val_loader, loss_fn, config, OUTPUT_DIR)
print(f'Stage 1 done in {(time.time()-t0)/60:.1f} min')

In [ ]:
config['epochs_stage2'] = EPOCHS_STAGE2
config['lr_predictor'] = LEARNING_RATE

t0 = time.time()
model, log_stage2 = train_stage2(model, train_loader, val_loader, loss_fn, config, OUTPUT_DIR)
print(f'Stage 2 done in {(time.time()-t0)/60:.1f} min')

### Evaluation & Visualization

In [ ]:
metrics, predictions = evaluate(model, test_loader, config)
print('\nVT-NIRS Test Results')
print('=' * 50)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.4f}')
    else:
        print(f'  {k:30s}: {v}')

In [ ]:
if log_stage1 is not None:
    fig = plot_training_curves(log_stage1, save_path=f'{OUTPUT_DIR}/fig_training_curves.png')
    plt.show()
else:
    print('No training logs available.')

In [ ]:
ite_pred = predictions['ite'].squeeze()
fig = plot_ite_distribution(ite_pred, model_name='VT-NIRS',
                             save_path=f'{OUTPUT_DIR}/fig_ite_distribution.png')
plt.show()
print(f'Mean ITE: {ite_pred.mean():.4f} | NIRS beneficial: {(ite_pred>0).mean()*100:.1f}%')

In [ ]:
fig = plot_decomposed_ite_scatter(predictions, save_path=f'{OUTPUT_DIR}/fig_decomposed_ite.png')
plt.show()

#### Effect

In [ ]:
test_n = len(ite_pred)
test_idx = np.arange(len(X_scaled))[-test_n:]
X_test_baseline = X_scaled[test_idx, -1, :]

for feat_name, label in [('fio2_X', 'FiO2'), ('sofa_X', 'SOFA Score'), ('age_X', 'Age')]:
    idx = FEATURE_COLS.index(feat_name)
    fig = plot_subgroup_ite_trends(
        ite_pred, X_test_baseline[:, idx], f'{label} (baseline)',
        save_path=f'{OUTPUT_DIR}/fig_ite_trend_{feat_name}.png')
    plt.show()

### Baseline Comparisons (Custom T-Learner & Causal Forest)

In [ ]:
psm_stay_ids = [valid_ids[j] for j in psm_idx]
cov_indexed = df_cov.set_index('stay_id')
X_baseline_psm = cov_indexed.loc[psm_stay_ids, FEATURE_COLS].fillna(0).values

baseline_results = run_baselines(
    X_baseline_psm, W_psm, VFD_psm,
    n_estimators=500, random_state=RANDOM_STATE
)
print('Baseline models trained.')

### Ablation Study

In [ ]:
ABLATION_CKPT = f'{OUTPUT_DIR}/ablation_base/best_stage2.pth'

model_base = VTNIRSModel(
    n_covariates=N_COVARIATES, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
).to(DEVICE)

loss_fn_base = CensoringAwareAdversarialLoss(
    lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV,
    lambda_vfd=LAMBDA_VFD, lambda_consist=LAMBDA_CONSIST,
    lambda_gate=0.0, lambda_ipm=LAMBDA_IPM, lambda_dr=LAMBDA_DR,
)

config_base = config.copy()
config_base['save_dir'] = f'{OUTPUT_DIR}/ablation_base'
os.makedirs(config_base['save_dir'], exist_ok=True)

if os.path.exists(ABLATION_CKPT):
    print(f'Loading ablation model from {ABLATION_CKPT}')
    model_base.load_state_dict(torch.load(ABLATION_CKPT, map_location=DEVICE))
    metrics_base, preds_base = evaluate(model_base, test_loader, config_base)
else:
    print('Training VT-NIRS-base (no gate entropy)...')
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)
    model_base, _ = train_stage1(model_base, train_loader, val_loader, loss_fn_base, config_base, config_base['save_dir'])
    model_base, _ = train_stage2(model_base, train_loader, val_loader, loss_fn_base, config_base, config_base['save_dir'])
    metrics_base, preds_base = evaluate(model_base, test_loader, config_base)
    torch.save(model_base.state_dict(), ABLATION_CKPT)

print('\nVT-NIRS-base results:')
for k, v in sorted(metrics_base.items()):
    if isinstance(v, float): print(f'  {k}: {v:.4f}')

### eICU External Validation

In [ ]:
print('Running eICU BigQuery extraction...')
df_eicu     = extract_eicu_cohort()
df_eicu_tx  = assign_eicu_treatment(df_eicu)
df_eicu_cov = extract_eicu_covariates(df_eicu_tx)
df_eicu_vfd = compute_eicu_vfd28(df_eicu_tx)
df_eicu_temporal = extract_eicu_temporal(df_eicu_tx)
X_eicu_ts, W_eicu_ts, VFD_eicu_ts, D_eicu_ts, eicu_valid_ids = (
    build_eicu_temporal_sequences(
        df_eicu_temporal, df_eicu_tx, df_eicu_vfd, df_eicu_cov,
        seq_len=SEQUENCE_LEN))

print(f'\neICU cohort: {len(df_eicu_tx):,} patients')
print(f'  NIRS: {(df_eicu_tx.Treatment_W==1).sum():,}')
print(f'  IMV:  {(df_eicu_tx.Treatment_W==0).sum():,}')
print(f'  VFD-28 mean: {df_eicu_vfd.vfd28.mean():.1f}')

if X_eicu_ts.shape[0] > 0:
    X_eicu_scaled, eicu_pad_masks, _ = normalize_and_mask(X_eicu_ts, N_TEMPORAL_COVARIATES)
    print(f'\neICU temporal ready: {X_eicu_scaled.shape}')
else:
    print('WARNING: No eICU temporal sequences built')

In [ ]:
df_eicu_cov_std, _ = standardize_features(df_eicu_cov, reference_df=df_cov)
eicu_tx_ids = set(df_eicu_vfd['stay_id'].values)
df_eicu_cov_tx = df_eicu_cov_std[df_eicu_cov_std['stay_id'].isin(eicu_tx_ids)].copy()
df_eicu_merged = df_eicu_cov_tx.merge(
    df_eicu_vfd[['stay_id', 'Treatment_W', 'vfd28']], on='stay_id', how='inner')

X_eicu_baseline = df_eicu_merged[FEATURE_COLS].fillna(0).values
W_eicu_bl = df_eicu_merged['Treatment_W'].values
VFD_eicu_bl = df_eicu_merged['vfd28'].values

print(f'eICU validation: {len(W_eicu_bl):,} patients')

eicu_psm_idx, _, _ = propensity_score_match_baseline(
    X_eicu_baseline, W_eicu_bl, FEATURE_COLS,
    caliper_scale=0.2, random_state=RANDOM_STATE)
X_eicu_psm = X_eicu_baseline[eicu_psm_idx]
W_eicu_psm = W_eicu_bl[eicu_psm_idx]
VFD_eicu_psm = VFD_eicu_bl[eicu_psm_idx]

print(f'\neICU post-PSM: NIRS VFD-28={VFD_eicu_psm[W_eicu_psm==1].mean():.2f}, '
      f'IMV VFD-28={VFD_eicu_psm[W_eicu_psm==0].mean():.2f}')

eicu_baseline_results = run_baselines(
    X_eicu_psm, W_eicu_psm, VFD_eicu_psm, random_state=RANDOM_STATE)

for name, res in eicu_baseline_results.items():
    ite = res['ite']
    print(f'  {name}: ATE={ite.mean():.4f}, std={ite.std():.4f}, '
          f'NIRS beneficial={(ite>0).mean()*100:.1f}%')

if X_eicu_ts.shape[0] > 0:
    print('\n' + '=' * 55)
    print('  VT-NIRS on eICU (transported validation)')
    print('=' * 55)
    from torch.utils.data import DataLoader
    eicu_dataset = NIRSTwinDataset(
        sequences=X_eicu_scaled, treatments=W_eicu_ts,
        vfd_observed=VFD_eicu_ts, delta=D_eicu_ts,
        pad_masks=eicu_pad_masks)
    eicu_loader = DataLoader(eicu_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)
    model.eval()
    eicu_vtnirs_metrics, eicu_vtnirs_preds = evaluate(model, eicu_loader, config)
    eicu_ite = eicu_vtnirs_preds['ite'].squeeze()

    print(f'\nVT-NIRS on eICU: ATE={eicu_ite.mean():.4f}, std={eicu_ite.std():.4f}, '
          f'NIRS beneficial={(eicu_ite>0).mean()*100:.1f}%')

    print('\n' + '=' * 55)
    print('  ALL METHODS COMPARISON')
    print('=' * 55)
    for name, tau in [('T-Learner', eicu_baseline_results['T-Learner']['ite']),
                      ('Causal Forest', eicu_baseline_results['Causal Forest']['ite']),
                      ('VT-NIRS (ours)', eicu_ite)]:
        print(f'  {name}: ATE={tau.mean():.4f}, NIRS beneficial={(tau>0).mean()*100:.1f}%')
else:
    print('\nSkipping VT-NIRS on eICU -- no temporal sequences available')

### Model Comparison

In [ ]:
test_n = len(ite_pred)
test_idx = np.arange(len(W_psm))[-test_n:]

tl_ite_test = baseline_results['T-Learner']['ite'][test_idx]
cf_ite_test = baseline_results['Causal Forest']['ite'][test_idx]

W_test = W_psm[test_idx]
VFD_test = VFD_psm[test_idx]

def policy_value(ite, W, VFD):
    recommend_nirs = (ite > 0).astype(float)
    match = (recommend_nirs == W).astype(float)
    return float(np.mean(match * VFD))

all_results = {
    'T-Learner': {'mean': policy_value(tl_ite_test, W_test, VFD_test), 'std': 0.0},
    'Causal Forest': {'mean': policy_value(cf_ite_test, W_test, VFD_test), 'std': 0.0},
    'VT-NIRS-base': {'mean': float(metrics_base.get('vfd_model_policy', 0)), 'std': 0.0},
    'VT-NIRS': {'mean': float(metrics.get('vfd_model_policy', 0)), 'std': 0.0},
}

fig = plot_model_comparison_bars(
    all_results, 'vfd_model_policy', 'Policy Value (VFD-28 days)',
    save_path=f'{OUTPUT_DIR}/fig_model_comparison.png')
plt.show()

### Results Visualizations

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from matplotlib.patches import FancyBboxPatch, Patch
from scipy import stats

psm_stay_ids = np.array([valid_ids[j] for j in psm_idx])
df_cov_idx = df_cov.set_index('stay_id')
df_full = df_cov_idx.loc[psm_stay_ids, FEATURE_COLS].reset_index()

df_full['Treatment_W'] = W_psm
df_full['vfd28'] = VFD_psm
df_full['delta'] = D_psm

df_full['ipcw_weight'] = 1.0

from torch.utils.data import DataLoader
full_dataset = NIRSTwinDataset(
    sequences=X_scaled, treatments=W_psm,
    vfd_observed=VFD_psm, delta=D_psm,
    pad_masks=pad_masks)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
model.eval()
_, full_preds = evaluate(model, full_loader, config)
df_full['tau_dr'] = full_preds['ite'].squeeze()

X_bl_full = df_full[FEATURE_COLS].fillna(0).values
bl_full = run_baselines(X_bl_full, W_psm, VFD_psm, random_state=RANDOM_STATE)
df_full['tau_tl'] = bl_full['T-Learner']['ite']
df_full['tau_cf'] = bl_full['Causal Forest']['ite']

out_path = f'{OUTPUT_DIR}/model_ite_results.csv'
df_full.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'  Shape: {df_full.shape}  ({len(df_full):,} patients x {df_full.shape[1]} columns)')
print(f'  Columns: {list(df_full.columns)}')
print(f'  VT-NIRS   (tau_dr): ATE={df_full["tau_dr"].mean():+.4f}, benefit={(df_full["tau_dr"]>0).mean()*100:.1f}%')
print(f'  T-Learner (tau_tl): ATE={df_full["tau_tl"].mean():+.4f}')
print(f'  Causal Forest (tau_cf): ATE={df_full["tau_cf"].mean():+.4f}')

#### Subgroup

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8))
groups = []
for label, lo, hi in [('SOFA 0-3 (Low)', -1, 3), ('SOFA 4-6 (Moderate)', 3, 6),
                        ('SOFA 7-10 (High)', 6, 10), ('SOFA >10 (Very High)', 10, 99)]:
    sub = df_full[(df_full['sofa_X'] > lo) & (df_full['sofa_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'SOFA'))
for label, lo, hi in [('Age 18-40', 17, 40), ('Age 41-60', 40, 60),
                        ('Age 61-75', 60, 75), ('Age >75', 75, 120)]:
    sub = df_full[(df_full['age_X'] > lo) & (df_full['age_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'Age'))
for label, lo, hi in [('P/F <100 (Severe ARDS)', 0, 100), ('P/F 100-200 (Moderate)', 100, 200),
                        ('P/F 200-300 (Mild)', 200, 300), ('P/F >300 (No ARDS)', 300, 9999)]:
    sub = df_full[(df_full['pf_ratio_X'] > lo) & (df_full['pf_ratio_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'P/F Ratio'))
for col, name in [('copd_X','COPD'),('chf_X','CHF'),('sepsis_X','Sepsis'),('immunosuppressed_X','Immunosuppressed')]:
    for val, suffix in [(1, ' (+)'), (0, ' (-)')]:
        sub = df_full[df_full[col]==val]
        groups.append((name+suffix, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'Comorbidity'))
groups.append((f'Overall Cohort', df_full['tau_dr'].mean(), df_full['tau_dr'].std()/np.sqrt(len(df_full)), len(df_full), 'Overall Cohort'))

labels = [g[0] for g in groups]
means = [g[1] for g in groups]
ses = [g[2] for g in groups]
ns = [g[3] for g in groups]
cats = [g[4] for g in groups]
colors_map = {'SOFA': '#2196F3', 'Age': '#4CAF50', 'P/F Ratio': '#FF9800', 'Comorbidity': '#9C27B0', 'Overall Cohort': '#E53935'}
colors = [colors_map[c] for c in cats]
y_pos = np.arange(len(groups))

x_min = min(m - 1.96*s for m, s in zip(means, ses))
x_max = max(m + 1.96*s for m, s in zip(means, ses))
x_pad = (x_max - x_min) * 0.15

ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.barh(y_pos, means, xerr=[1.96*s for s in ses], height=0.6, color=colors,
        alpha=0.7, edgecolor='black', linewidth=0.5, capsize=3)

x_text = x_max + x_pad * 0.3
for i, n in enumerate(ns):
    ax.text(x_text, i, f'n={n:,}', va='center', fontsize=9)

ax.set_xlim(x_min - x_pad, x_max + x_pad * 1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Estimated ITE (VFD-28 days, positive = NIRS beneficial)', fontsize=11)

prev_cat = None
for i, cat in enumerate(cats):
    if prev_cat and cat != prev_cat:
        ax.axhline(y=i-0.5, linestyle='-', linewidth=0.5, alpha=0.5)
    prev_cat = cat

ax.legend(handles=[Patch(facecolor=v, label=k, alpha=0.7) for k,v in colors_map.items()],
          loc='lower center', fontsize=8, ncol=len(colors_map))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_subgroup_forest.tif', dpi=300, bbox_inches='tight')
plt.show()

#### Waterfall plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_sorted = df_full.sort_values('tau_dr')
ites = df_sorted['tau_dr'].values; n = len(ites)
sofa = df_sorted['sofa_X'].values
colors_wf = np.where(sofa > 10, '#E53935', np.where(sofa > 6, '#FF9800', np.where(sofa > 3, '#FFC107', '#4CAF50')))
ax.bar(np.arange(n), ites, width=1.0, color=colors_wf, edgecolor='none', alpha=0.8)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
nirs_pct = (ites > 0).sum() / n * 100
ax.text(n*0.75, max(ites)*0.7, f'NIRS beneficial\n({nirs_pct:.2f}%)', fontsize=10, ha='center', color='#2E7D32', fontweight='bold')
ax.text(n*0.20, min(ites)*0.8, f'IMV beneficial\n({100-nirs_pct:.2f}%)', fontsize=10, ha='center', color='#C62828', fontweight='bold')
ax.set_xlabel('Patients (ranked by ITE)', fontsize=11); ax.set_ylabel('ITE (VFD-28 days)', fontsize=11)
ax.legend(handles=[Patch(facecolor=c, label=l) for c,l in [('#4CAF50','SOFA 0-3'),('#FFC107','SOFA 4-6'),('#FF9800','SOFA 7-10'),('#E53935','SOFA >10')]], 
          loc='center left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_waterfall_ite.tif', dpi=300, bbox_inches='tight')

plt.show()

#### SOFA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sofa_groups = [('0-3', 0, 3), ('4-6', 4, 6), ('7-10', 7, 10), ('>10', 11, 99)]
colors_sofa = ['#4CAF50', '#FFC107', '#FF9800', '#E53935']
recommend_nirs = df_full['tau_dr'] > 0; got_nirs = df_full['Treatment_W'] == 1

sofa_means, sofa_sems, sofa_labels_list = [], [], []
concordance_data = []
for label, lo, hi in sofa_groups:
    sub = df_full[(df_full['sofa_X'] >= lo) & (df_full['sofa_X'] <= hi)]
    sofa_means.append(sub['tau_dr'].mean())
    sofa_sems.append(1.96 * sub['tau_dr'].std() / np.sqrt(len(sub)))
    sofa_labels_list.append(f'SOFA {label}\n(n={len(sub)})')
    rec = recommend_nirs[sub.index]; got = got_nirs[sub.index]
    conc = (rec == got).mean() * 100
    conc_mask = rec == got; disc_mask = ~conc_mask
    vfd_c = sub.loc[conc_mask, 'vfd28'].mean() if conc_mask.sum() > 0 else 0
    vfd_d = sub.loc[disc_mask, 'vfd28'].mean() if disc_mask.sum() > 0 else 0
    concordance_data.append((conc, vfd_c, vfd_d))

axes[0].bar(range(4), sofa_means, yerr=sofa_sems, color=colors_sofa, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(sofa_labels_list, fontsize=9)
axes[0].set_ylabel('Mean ITE (VFD-28 days)'); axes[0].set_title('(A) ITE by SOFA Severity', fontweight='bold')

conc_rates = [c[0] for c in concordance_data]
axes[1].bar(range(4), conc_rates, color=colors_sofa, edgecolor='black', linewidth=0.5, alpha=0.8)
axes[1].axhline(y=50, color='gray', linestyle=':', linewidth=1)
for i, v in enumerate(conc_rates): axes[1].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(sofa_labels_list, fontsize=9)
axes[1].set_ylabel('Concordance (%)'); axes[1].set_title('(B) Model-Clinician Concordance', fontweight='bold'); axes[1].set_ylim(0,100)

x = np.arange(4); w = 0.35
axes[2].bar(x-w/2, [c[1] for c in concordance_data], w, color='#2196F3', label='Concordant', edgecolor='black', linewidth=0.5)
axes[2].bar(x+w/2, [c[2] for c in concordance_data], w, color='#FF5722', label='Discordant', edgecolor='black', linewidth=0.5)
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(sofa_labels_list, fontsize=9)
axes[2].set_ylabel('Mean VFD-28 (days)'); axes[2].set_title('(C) Outcomes: Concordant vs Discordant', fontweight='bold'); axes[2].legend(fontsize=9)
plt.suptitle('Treatment Effect Heterogeneity Stratified by Organ Dysfunction Severity', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_sofa_stratified.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

ite_p90 = df_full['tau_dr'].quantile(0.90)
ite_p10 = df_full['tau_dr'].quantile(0.10)

caseA_mask = (df_full['sofa_X']<=5) & (df_full['age_X']<=50) & (df_full['tau_dr']>=ite_p90) & (df_full['Treatment_W']==1) & (df_full['delta']==1) & (df_full['vfd28']>20)
if caseA_mask.sum() > 0:
    caseA = df_full[caseA_mask].iloc[0]
else:
    caseA = df_full[(df_full['tau_dr']>=ite_p90) & (df_full['Treatment_W']==1) & (df_full['delta']==1)].iloc[0]

caseB_mask = (df_full['sofa_X']>8) & (df_full['tau_dr']<=ite_p10) & (df_full['Treatment_W']==0) & (df_full['delta']==1)
if caseB_mask.sum() > 0:
    caseB = df_full[caseB_mask].iloc[0]
else:
    caseB = df_full[(df_full['tau_dr']<=ite_p10) & (df_full['Treatment_W']==0) & (df_full['delta']==1)].iloc[0]

caseC_mask = (df_full['copd_X']==1) & (df_full['tau_dr']>df_full['tau_dr'].median()) & (df_full['Treatment_W']==1) & (df_full['delta']==1)
if caseC_mask.sum() > 0:
    caseC = df_full[caseC_mask].iloc[0]
else:
    caseC = df_full[(df_full['copd_X']==1) & (df_full['tau_dr']>0)].iloc[0]

caseD_mask = (df_full['tau_dr']>df_full['tau_dr'].median()) & (df_full['Treatment_W']==0) & (df_full['delta']==0)
if caseD_mask.sum() > 0:
    caseD = df_full[caseD_mask].iloc[0]
else:
    caseD = df_full[(df_full['tau_dr']>0) & (df_full['Treatment_W']==0)].iloc[0]

cases = [('Case A: NIRS Concordant (Survived)', caseA, '#4CAF50'),
         ('Case B: IMV Concordant (Survived)', caseB, '#2196F3'),
         ('Case C: COPD + NIRS Benefit', caseC, '#FF9800'),
         ('Case D: Model-Clinician Discordance', caseD, '#E53935')]

fig, axes_grid = plt.subplots(2, 2, figsize=(16, 14))
for idx, (title, patient, color) in enumerate(cases):
    ax = axes_grid.flatten()[idx]; ax.set_xlim(0,10); ax.set_ylim(0,10); ax.axis('off')
    ax.text(5, 9.7, title, fontsize=13, fontweight='bold', ha='center', va='top', color=color)
    gender = 'Male' if patient['gender_X']==1 else 'Female'
    y = 9.0
    for line in [f'{gender}, {patient["age_X"]:.0f}y, BMI {patient["bmi_X"]:.1f}',
                 f'SOFA {patient["sofa_X"]:.0f} | GCS {patient["gcs_X"]:.0f} | SAPS-II {patient["sapsii_X"]:.0f}',
                 f'HR {patient["hr_mean_X"]:.0f} | RR {patient["rr_mean_X"]:.0f} | SpO2 {patient["spo2_mean_X"]:.0f}% | MAP {patient["mbp_mean_X"]:.0f}',
                 f'PaO2 {patient["pao2_X"]:.0f} | PaCO2 {patient["paco2_X"]:.0f} | pH {patient["ph_X"]:.2f} | FiO2 {patient["fio2_X"]:.0f}%',
                 f'P/F: {patient["pf_ratio_X"]:.0f} | ROX: {patient["rox_index_X"]:.1f} | Lactate: {patient["lactate_X"]:.1f}']:
        ax.text(0.3, y, line, fontsize=9, va='top', fontfamily='monospace'); y -= 0.35
    comorbids = [n for n,c in [('COPD','copd_X'),('CHF','chf_X'),('Sepsis','sepsis_X')] if patient[c]==1]
    ax.text(0.3, y, f'Comorbidities: {", ".join(comorbids) if comorbids else "None"}', fontsize=9, va='top', fontfamily='monospace')
    y -= 0.6
    tx = 'NIRS' if patient['Treatment_W']==1 else 'IMV'
    outcome = 'Survived' if patient['delta']==1 else 'Died'
    oc = '#2E7D32' if patient['delta']==1 else '#C62828'
    ax.text(0.3, y, f'Received: {tx}', fontsize=10, va='top', fontweight='bold')
    ax.text(3.5, y, f'Outcome: {outcome}', fontsize=10, va='top', fontweight='bold', color=oc)
    ax.text(6.5, y, f'VFD-28: {patient["vfd28"]:.1f}d', fontsize=10, va='top', fontweight='bold')
    y -= 0.6
    rec = 'NIRS' if patient['tau_dr']>0 else 'IMV'
    rc = '#2E7D32' if rec==tx else '#C62828'
    match = '(Concordant)' if rec==tx else '(DISCORDANT)'
    ax.text(0.3, y, f'VT-NIRS: {patient["tau_dr"]:+.2f}', fontsize=10, va='top')
    ax.text(3.5, y, f'T-Learner: {patient["tau_tl"]:+.2f}', fontsize=10, va='top')
    ax.text(6.5, y, f'CF: {patient["tau_cf"]:+.2f}', fontsize=10, va='top')
    y -= 0.4
    ax.text(0.3, y, f'Recommendation: {rec} {match}', fontsize=10, va='top', fontweight='bold', color=rc)
    rect = FancyBboxPatch((0.05,0.1), 9.9, 9.5, boxstyle='round,pad=0.1', linewidth=2, edgecolor=color, facecolor='white', alpha=0.3)
    ax.add_patch(rect)
plt.suptitle('Representative Patient Case Studies', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_patient_cases.png', dpi=200, bbox_inches='tight'); plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

patients_list = [
    ('Patient 1', caseA, '#4CAF50', '-',  'NIRS Concordant'),
    ('Patient 2', caseB, '#2196F3', '--', 'IMV Concordant'),
    ('Patient 3', caseC, '#FF9800', '-.', 'COPD + NIRS'),
    ('Patient 4', caseD, '#E53935', ':',  'Discordant'),
]

radar_features = ['sofa_X', 'sapsii_X', 'pf_ratio_X', 'rox_index_X', 'lactate_X', 'fio2_X']
radar_labels   = ['SOFA',   'SAPS-II',  'P/F Ratio',  'ROX Index',   'Lactate',    'FiO2']
feat_ranges = {
    'sofa_X': (0, 20), 'sapsii_X': (0, 80), 'pf_ratio_X': (50, 300),
    'rox_index_X': (2, 10), 'lactate_X': (0, 5), 'fio2_X': (21, 100)
}

fig = plt.figure(figsize=(15, 7))

ax_radar = fig.add_subplot(121, polar=True)
n_feat = len(radar_features)
angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

for pname, pdata, pcolor, pstyle, plabel in patients_list:
    vals_raw = []
    vals_norm = []
    for f in radar_features:
        lo, hi = feat_ranges[f]
        raw = pdata[f]
        vals_raw.append(raw)
        vals_norm.append(np.clip((raw - lo) / (hi - lo), 0, 1))
    vals_norm += vals_norm[:1]
    ax_radar.plot(angles, vals_norm, linestyle=pstyle, linewidth=2.0, markersize=5,
                  marker='o', color=pcolor, label=pname, alpha=0.85)
    ax_radar.fill(angles, vals_norm, color=pcolor, alpha=0.05)
    for k, (ang, vn, vr) in enumerate(zip(angles[:-1], vals_norm[:-1], vals_raw)):
        offset_r = 0.08
        fmt = f'{vr:.0f}' if vr >= 1 else f'{vr:.1f}'
        ax_radar.annotate(fmt, xy=(ang, vn + offset_r), fontsize=6,
                          color=pcolor, ha='center', va='center', fontweight='bold')

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, fontsize=9)
ax_radar.set_ylim(0, 1.15)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7, color='grey')
ax_radar.set_title('(A) Clinical Profiles', fontsize=11, fontweight='bold', pad=25)

ax_bar = fig.add_subplot(122)
methods = ['tau_dr', 'tau_tl', 'tau_cf']
method_labels = ['VT-NIRS (DR-IPCW)', 'T-Learner', 'Causal Forest']
method_colors = ['#1565C0', '#43A047', '#EF6C00']
pnames = [p[0] for p in patients_list]
n_p = len(pnames)
bar_width = 0.22
y_pos = np.arange(n_p)

for j, (mcol, mlabel, mcolor) in enumerate(zip(methods, method_labels, method_colors)):
    vals = [patients_list[i][1][mcol] for i in range(n_p)]
    offset = (j - 1) * bar_width
    bars = ax_bar.barh(y_pos + offset, vals, height=bar_width,
                       label=mlabel, alpha=0.85, color=mcolor,
                       edgecolor='black', linewidth=0.4)
    for bar, v in zip(bars, vals):
        x_pos = v + 0.3 if v >= 0 else v - 0.3
        ha = 'left' if v >= 0 else 'right'
        ax_bar.text(x_pos, bar.get_y() + bar.get_height() / 2,
                    f'{v:.1f}', va='center', ha=ha, fontsize=7, fontweight='bold')

ax_bar.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
xlim = ax_bar.get_xlim()
ax_bar.axvspan(0, max(xlim[1], 5), alpha=0.04, color='green')
ax_bar.axvspan(min(xlim[0], -5), 0, alpha=0.04, color='red')
ax_bar.text(max(xlim[1], 5) * 0.7, n_p - 0.1, 'NIRS benefit', fontsize=7,
            color='green', alpha=0.6, ha='center', style='italic')
ax_bar.text(min(xlim[0], -5) * 0.7, n_p - 0.1, 'IMV benefit', fontsize=7,
            color='red', alpha=0.6, ha='center', style='italic')

ylabels = []
for pname, pdata, pcolor, pstyle, plabel in patients_list:
    tx = 'NIRS' if pdata['Treatment_W'] == 1 else 'IMV'
    out = 'Survived' if pdata['delta'] == 1 else 'Died'
    vfd = pdata['vfd28']
    ylabels.append(f'{pname}\n({plabel})\n{tx}, {out}, VFD={vfd:.0f}')

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(ylabels, fontsize=8)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Estimated ITE (VFD-28 days)', fontsize=10)
ax_bar.set_title('(B) ITE Estimates by Method', fontsize=11, fontweight='bold')
ax_bar.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15),
              ncol=3, fontsize=8, frameon=True)

ax_radar.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15),
                ncol=2, fontsize=8, frameon=True)

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig(f'{OUTPUT_DIR}/fig_patient_radar_ite.tif', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(df_full['tau_dr'], df_full['tau_tl'], alpha=0.2, s=8, c='#2196F3', edgecolors='none')
axes[0].axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6); axes[0].axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
agr1 = ((df_full['tau_dr']>0)==(df_full['tau_tl']>0)).mean()*100
axes[0].set_title(f'(A) VT-NIRS vs T-Learner\nAgreement: {agr1:.0f}%', fontweight='bold')
axes[0].set_xlabel('VT-NIRS ITE'); axes[0].set_ylabel('T-Learner ITE')

axes[1].scatter(df_full['tau_dr'], df_full['tau_cf'], alpha=0.2, s=8, c='#FF9800', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6); axes[1].axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
agr2 = ((df_full['tau_dr']>0)==(df_full['tau_cf']>0)).mean()*100
axes[1].set_title(f'(B) VT-NIRS vs Causal Forest\nAgreement: {agr2:.0f}%', fontweight='bold')
axes[1].set_xlabel('VT-NIRS ITE'); axes[1].set_ylabel('Causal Forest ITE')

sofa_grps = [('0-3',0,3),('4-6',4,6),('7-10',7,10),('>10',11,99)]
x = np.arange(4); w = 0.25
for j,(col,nm,clr) in enumerate([('tau_dr','VT-NIRS','#2196F3'),('tau_tl','T-Learner','#4CAF50'),('tau_cf','Causal Forest','#FF9800')]):
    pcts = [(df_full[(df_full['sofa_X']>=lo)&(df_full['sofa_X']<=hi)][col]>0).mean()*100 for _,lo,hi in sofa_grps]
    axes[2].bar(x+(j-1)*w, pcts, w, color=clr, label=nm, edgecolor='black', linewidth=0.5, alpha=0.8)
axes[2].axhline(50, color='gray', linestyle=':', linewidth=1)
axes[2].set_xticks(x); axes[2].set_xticklabels([f'SOFA {l}' for l,_,_ in sofa_grps], fontsize=9)
axes[2].set_ylabel('% Recommended NIRS'); axes[2].set_title('(C) NIRS Rate by SOFA Across Methods', fontweight='bold'); axes[2].legend(fontsize=9); axes[2].set_ylim(0,105)
plt.suptitle('Cross-Method ITE Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_method_comparison_detailed.png', dpi=200, bbox_inches='tight'); plt.show()

#### Survival vs. Ventilation Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sofa_bins = [0,3,6,10,30]; sofa_lab = ['0-3','4-6','7-10','>10']
pf_bins = [0,100,200,300,1000]; pf_lab = ['<100','100-200','200-300','>300']
df_full['sofa_bin'] = pd.cut(df_full['sofa_X'], bins=sofa_bins, labels=sofa_lab)
df_full['pf_bin'] = pd.cut(df_full['pf_ratio_X'], bins=pf_bins, labels=pf_lab)
piv_ite = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc='mean')
piv_n = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc='count')

ite_abs_max = max(abs(piv_ite.values[~np.isnan(piv_ite.values)].min()), abs(piv_ite.values[~np.isnan(piv_ite.values)].max()))
ite_vbound = max(ite_abs_max * 1.2, 0.5)

im = axes[0].imshow(piv_ite.values, cmap='RdYlGn', aspect='auto', vmin=-ite_vbound, vmax=ite_vbound)
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(sofa_lab)
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(pf_lab)
axes[0].set_xlabel('SOFA Score'); axes[0].set_ylabel('P/F Ratio')
axes[0].set_title('(A) Mean ITE by SOFA x P/F', fontweight='bold')
for i in range(4):
    for j in range(4):
        v = piv_ite.values[i,j]; n = piv_n.values[i,j]
        if not np.isnan(v):
            tc = 'white' if abs(v) > ite_vbound * 0.6 else 'black'
            axes[0].text(j, i, f'{v:+.1f}\nn={int(n)}', ha='center', va='center', fontsize=9, fontweight='bold', color=tc)
plt.colorbar(im, ax=axes[0], label='Mean ITE (VFD-28 days)')

piv_pct = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc=lambda x: (x>0).mean()*100)
im2 = axes[1].imshow(piv_pct.values, cmap='RdYlGn', aspect='auto', vmin=20, vmax=80)
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(sofa_lab)
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(pf_lab)
axes[1].set_xlabel('SOFA Score'); axes[1].set_ylabel('P/F Ratio')
axes[1].set_title('(B) % NIRS Beneficial by SOFA x P/F', fontweight='bold')
for i in range(4):
    for j in range(4):
        v = piv_pct.values[i,j]; n = piv_n.values[i,j]
        if not np.isnan(v): axes[1].text(j, i, f'{v:.0f}%\nn={int(n)}', ha='center', va='center', fontsize=9, fontweight='bold', color='white' if v>70 or v<30 else 'black')
plt.colorbar(im2, ax=axes[1], label='% NIRS Beneficial')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_clinical_decision_matrix.tif', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rox_bins = [0,4,6,8,10,50]; rox_labs = ['<4\n(High fail)','4-6','6-8','8-10','>10\n(Low fail)']
df_full['rox_bin'] = pd.cut(df_full['rox_index_X'], bins=rox_bins, labels=rox_labs)
rox_m, rox_se, rox_n = [], [], []
for lab in rox_labs:
    sub = df_full[df_full['rox_bin']==lab]
    rox_m.append(sub['tau_dr'].mean()); rox_se.append(1.96*sub['tau_dr'].std()/np.sqrt(max(len(sub),1))); rox_n.append(len(sub))
clrs_rox = ['#E53935','#FF9800','#FFC107','#8BC34A','#4CAF50']
axes[0].bar(range(5), rox_m, yerr=rox_se, color=clrs_rox, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xticks(range(5)); axes[0].set_xticklabels(rox_labs, fontsize=8)
for i,n in enumerate(rox_n): axes[0].text(i, rox_m[i]+rox_se[i]+0.1, f'n={n}', ha='center', fontsize=8, color='gray')
axes[0].set_ylabel('Mean ITE (VFD-28 days)'); axes[0].set_xlabel('ROX Index')
axes[0].set_title('(A) ITE by ROX Index', fontweight='bold')

pf_cuts = [0,100,150,200,250,300,500]; pf_labs = ['<100','100-150','150-200','200-250','250-300','>300']
df_full['pf_det'] = pd.cut(df_full['pf_ratio_X'], bins=pf_cuts, labels=pf_labs)
pf_m, pf_se, pf_n = [], [], []
for lab in pf_labs:
    sub = df_full[df_full['pf_det']==lab]
    pf_m.append(sub['tau_dr'].mean()); pf_se.append(1.96*sub['tau_dr'].std()/np.sqrt(max(len(sub),1))); pf_n.append(len(sub))
clrs_pf = ['#E53935','#FF5722','#FF9800','#FFC107','#8BC34A','#4CAF50']
axes[1].bar(range(6), pf_m, yerr=pf_se, color=clrs_pf, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(pf_labs, fontsize=8)
for i,n in enumerate(pf_n): axes[1].text(i, pf_m[i]+pf_se[i]+0.1, f'n={n}', ha='center', fontsize=8, color='gray')
axes[1].set_ylabel('Mean ITE (VFD-28 days)'); axes[1].set_xlabel('P/F Ratio')
axes[1].set_title('(B) ITE by P/F Ratio', fontweight='bold')

nirs_ben = df_full[df_full['tau_dr']>0]; imv_ben = df_full[df_full['tau_dr']<=0]
cvars = ['sofa_X','age_X','pf_ratio_X','rox_index_X','hr_mean_X','rr_mean_X','lactate_X']
cvlabs = ['SOFA','Age','P/F Ratio','ROX Index','Heart Rate','Resp Rate','Lactate']
x_pos = np.arange(len(cvars)); w = 0.35
axes[2].barh(x_pos-w/2, [nirs_ben[v].mean() for v in cvars], w, color='#4CAF50', label='NIRS beneficial', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[2].barh(x_pos+w/2, [imv_ben[v].mean() for v in cvars], w, color='#E53935', label='IMV beneficial', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[2].set_yticks(x_pos); axes[2].set_yticklabels(cvlabs, fontsize=9)
axes[2].set_xlabel('Mean Value'); axes[2].set_title('(C) NIRS vs IMV Beneficial', fontweight='bold'); axes[2].legend(fontsize=9)

plt.suptitle('Respiratory-Specific Treatment Effect Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_respiratory_stratified.png', dpi=200, bbox_inches='tight'); plt.show()

#### Recommendation 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
recommend_nirs = df_full['tau_dr'] > 0; got_nirs = df_full['Treatment_W'] == 1
concordant = recommend_nirs == got_nirs
conc_vfd = df_full.loc[concordant, 'vfd28']; disc_vfd = df_full.loc[~concordant, 'vfd28']
axes[0].hist(conc_vfd, bins=30, alpha=0.6, color='#2196F3', label=f'Concordant (n={len(conc_vfd)})', density=True)
axes[0].hist(disc_vfd, bins=30, alpha=0.6, color='#FF5722', label=f'Discordant (n={len(disc_vfd)})', density=True)
axes[0].axvline(conc_vfd.mean(), color='#1565C0', linestyle='--', linewidth=2, label=f'Conc mean: {conc_vfd.mean():.1f}d')
axes[0].axvline(disc_vfd.mean(), color='#BF360C', linestyle='--', linewidth=2, label=f'Disc mean: {disc_vfd.mean():.1f}d')
axes[0].set_xlabel('VFD-28 (days)'); axes[0].set_ylabel('Density'); axes[0].set_title('(A) VFD-28 Concordant vs Discordant', fontweight='bold'); axes[0].legend(fontsize=8)

sofa_grps = [('0-3',0,3),('4-6',4,6),('7-10',7,10),('>10',11,99)]
x = np.arange(4); w = 0.35; mort_c, mort_d = [], []
for _,lo,hi in sofa_grps:
    sub = df_full[(df_full['sofa_X']>=lo)&(df_full['sofa_X']<=hi)]
    cc = sub[concordant[sub.index]]; dd = sub[~concordant[sub.index]]
    mort_c.append((1-cc['delta'].mean())*100 if len(cc)>0 else 0)
    mort_d.append((1-dd['delta'].mean())*100 if len(dd)>0 else 0)
axes[1].bar(x-w/2, mort_c, w, color='#2196F3', label='Concordant', edgecolor='black', linewidth=0.5)
axes[1].bar(x+w/2, mort_d, w, color='#FF5722', label='Discordant', edgecolor='black', linewidth=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'SOFA {l}' for l,_,_ in sofa_grps])
axes[1].set_ylabel('Mortality (%)'); axes[1].set_title('(B) Mortality: Concordant vs Discordant', fontweight='bold'); axes[1].legend(fontsize=9)

df_s = df_full.copy(); df_s['abs_ite'] = df_s['tau_dr'].abs(); df_s = df_s.sort_values('abs_ite', ascending=False)
df_s['cum_ben'] = (df_s['tau_dr']*(df_s['tau_dr']>0)*(df_s['Treatment_W']==1)).cumsum() + (-df_s['tau_dr']*(df_s['tau_dr']<0)*(df_s['Treatment_W']==0)).cumsum()
df_s['pct_pts'] = np.arange(1, len(df_s)+1)/len(df_s)*100
axes[2].plot(df_s['pct_pts'], df_s['cum_ben'], color='#2196F3', linewidth=2)
axes[2].fill_between(df_s['pct_pts'], 0, df_s['cum_ben'], alpha=0.2, color='#2196F3')
axes[2].set_xlabel('% Patients (ranked by |ITE|)'); axes[2].set_ylabel('Cumulative VFD-28 Benefit')
axes[2].set_title('(C) Cumulative Benefit Curve', fontweight='bold')
plt.suptitle('Treatment Recommendation Validation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_outcome_validation.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

nirs_pts = df_full[df_full['Treatment_W']==1]
high = nirs_pts[nirs_pts['tau_dr']>0]; low = nirs_pts[nirs_pts['tau_dr']<=0]
for grp, data, clr, sty in [(f'NIRS Recommended (n={len(high)})', high, '#4CAF50', '-'),
                              (f'IMV Recommended (n={len(low)})', low, '#E53935', '--')]:
    sv = np.sort(data['vfd28'].values)
    sf = 1 - np.arange(1, len(sv)+1)/len(sv)
    axes[0].step(sv, sf, where='post', color=clr, linestyle=sty, linewidth=2, label=grp)
axes[0].axvline(high['vfd28'].mean(), color='#4CAF50', linestyle=':', alpha=0.5)
axes[0].axvline(low['vfd28'].mean(), color='#E53935', linestyle=':', alpha=0.5)
axes[0].text(high['vfd28'].mean()+0.3, 0.95, f'Mean: {high["vfd28"].mean():.1f}d', fontsize=8, color='#4CAF50')
axes[0].text(low['vfd28'].mean()+0.3, 0.90, f'Mean: {low["vfd28"].mean():.1f}d', fontsize=8, color='#E53935')
axes[0].set_xlabel('VFD-28 (days)'); axes[0].set_ylabel('Fraction >= x')
axes[0].set_title('(A) NIRS Recipients by Predicted Benefit', fontweight='bold'); axes[0].legend(fontsize=9)

imv_pts = df_full[df_full['Treatment_W']==0]
hi_imv = imv_pts[imv_pts['tau_dr']<=0]; lo_imv = imv_pts[imv_pts['tau_dr']>0]
for grp, data, clr, sty in [(f'IMV Recommended (n={len(hi_imv)})', hi_imv, '#2196F3', '-'),
                              (f'NIRS Recommended (n={len(lo_imv)})', lo_imv, '#FF9800', '--')]:
    sv = np.sort(data['vfd28'].values)
    sf = 1 - np.arange(1, len(sv)+1)/len(sv)
    axes[1].step(sv, sf, where='post', color=clr, linestyle=sty, linewidth=2, label=grp)
axes[1].set_xlabel('VFD-28 (days)'); axes[1].set_ylabel('Fraction >= x')
axes[1].set_title('(B) IMV Recipients by Predicted Benefit', fontweight='bold'); axes[1].legend(fontsize=9)

_, p_nirs = stats.mannwhitneyu(high['vfd28'], low['vfd28'], alternative='greater')
_, p_imv = stats.mannwhitneyu(hi_imv['vfd28'], lo_imv['vfd28'], alternative='greater')
plt.suptitle('VFD-28 Survival Curves by Predicted Treatment Benefit', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_km_ite_stratified.png', dpi=200, bbox_inches='tight'); plt.show()
print(f'NIRS recipients: Mann-Whitney p = {p_nirs:.2e}')
print(f'IMV recipients: Mann-Whitney p = {p_imv:.2e}')

In [ ]:
pred = pred_ens
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
vfd_imv = pred['vfd_0'].squeeze(); vfd_nirs = pred['vfd_1'].squeeze(); ite_vals = pred['ite'].squeeze()
colors_sc = np.where(ite_vals > 0, '#4CAF50', '#E53935')

axes[0,0].scatter(vfd_imv, vfd_nirs, c=colors_sc, alpha=0.4, s=15, edgecolors='none')
axes[0,0].plot([0,28],[0,28],'k--',linewidth=1,alpha=0.5,label='Equal outcome')
axes[0,0].set_xlabel('Predicted VFD-28 under IMV'); axes[0,0].set_ylabel('Predicted VFD-28 under NIRS')
axes[0,0].set_title('(A) Counterfactual VFD-28: NIRS vs IMV', fontweight='bold'); axes[0,0].legend(fontsize=9)

p_s0 = pred['p_surv_0'].squeeze(); p_s1 = pred['p_surv_1'].squeeze()
axes[0,1].scatter(p_s0, p_s1, c=colors_sc, alpha=0.4, s=15, edgecolors='none')
axes[0,1].plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5,label='Equal survival')
axes[0,1].set_xlabel('P(survive) under IMV'); axes[0,1].set_ylabel('P(survive) under NIRS')
axes[0,1].set_title('(B) Counterfactual Survival Probability', fontweight='bold'); axes[0,1].legend(fontsize=9)

vc0 = pred['vfd_cond_0'].squeeze(); vc1 = pred['vfd_cond_1'].squeeze()
axes[1,0].scatter(vc0, vc1, c=colors_sc, alpha=0.4, s=15, edgecolors='none')
axes[1,0].plot([0,28],[0,28],'k--',linewidth=1,alpha=0.5,label='Equal VFD|survive')
axes[1,0].set_xlabel('E[VFD|survived] under IMV'); axes[1,0].set_ylabel('E[VFD|survived] under NIRS')
axes[1,0].set_title('(C) Conditional VFD (Given Survival)', fontweight='bold'); axes[1,0].legend(fontsize=9)

ite_surv = pred['ite_survival'].squeeze(); ite_vfd = pred['ite_vfd_cond'].squeeze()
sort_idx = np.argsort(ite_vals); step = max(1, len(ite_vals)//100); samp = sort_idx[::step]
xb = np.arange(len(samp))
axes[1,1].bar(xb, ite_surv[samp], color='#2196F3', alpha=0.7, label='Survival component')
axes[1,1].bar(xb, ite_vfd[samp], bottom=ite_surv[samp], color='#FF9800', alpha=0.7, label='VFD|survived component')
axes[1,1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1,1].set_xlabel('Patients (sorted by ITE)'); axes[1,1].set_ylabel('ITE Decomposition')
axes[1,1].set_title('(D) ITE: Survival vs Ventilation Components', fontweight='bold'); axes[1,1].legend(fontsize=9)

plt.suptitle('Counterfactual Outcome Profiles (VT-NIRS Ensemble, N=789)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_counterfactual_profiles.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

patients_list = [
    ('Patient 1', caseA, '#4CAF50', '-',  'NIRS Concordant'),
    ('Patient 2', caseB, '#2196F3', '--', 'IMV Concordant'),
    ('Patient 3', caseC, '#FF9800', '-.', 'COPD + NIRS'),
    ('Patient 4', caseD, '#E53935', ':',  'Discordant'),
]

radar_features = ['sofa_X', 'sapsii_X', 'pf_ratio_X', 'rox_index_X', 'lactate_X', 'fio2_X']
radar_labels   = ['SOFA',   'SAPS-II',  'P/F Ratio',  'ROX Index',   'Lactate',    'FiO2']
feat_ranges = {
    'sofa_X': (0, 20), 'sapsii_X': (0, 80), 'pf_ratio_X': (50, 300),
    'rox_index_X': (2, 10), 'lactate_X': (0, 5), 'fio2_X': (21, 100)
}

fig = plt.figure(figsize=(15, 7))

ax_radar = fig.add_subplot(121, polar=True)
n_feat = len(radar_features)
angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

for pname, pdata, pcolor, pstyle, plabel in patients_list:
    vals_raw = []
    vals_norm = []
    for f in radar_features:
        lo, hi = feat_ranges[f]
        raw = pdata[f]
        vals_raw.append(raw)
        vals_norm.append(np.clip((raw - lo) / (hi - lo), 0, 1))
    vals_norm += vals_norm[:1]
    ax_radar.plot(angles, vals_norm, linestyle=pstyle, linewidth=2.0, markersize=5,
                  marker='o', color=pcolor, label=pname, alpha=0.85)
    ax_radar.fill(angles, vals_norm, color=pcolor, alpha=0.05)
    for k, (ang, vn, vr) in enumerate(zip(angles[:-1], vals_norm[:-1], vals_raw)):
        offset_r = 0.08
        fmt = f'{vr:.0f}' if vr >= 1 else f'{vr:.1f}'
        ax_radar.annotate(fmt, xy=(ang, vn + offset_r), fontsize=6,
                          color=pcolor, ha='center', va='center', fontweight='bold')

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, fontsize=9)
ax_radar.set_ylim(0, 1.15)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7, color='grey')
ax_radar.set_title('(A) Clinical Profiles', fontsize=11, fontweight='bold', pad=25)

ax_bar = fig.add_subplot(122)
methods = ['tau_dr', 'tau_tl', 'tau_cf']
method_labels = ['VT-NIRS (DR-IPCW)', 'T-Learner', 'Causal Forest']
method_colors = ['#1565C0', '#43A047', '#EF6C00']
pnames = [p[0] for p in patients_list]
n_p = len(pnames)
bar_width = 0.22
y_pos = np.arange(n_p)

for j, (mcol, mlabel, mcolor) in enumerate(zip(methods, method_labels, method_colors)):
    vals = [patients_list[i][1][mcol] for i in range(n_p)]
    offset = (j - 1) * bar_width
    bars = ax_bar.barh(y_pos + offset, vals, height=bar_width,
                       label=mlabel, alpha=0.85, color=mcolor,
                       edgecolor='black', linewidth=0.4)
    for bar, v in zip(bars, vals):
        x_pos = v + 0.3 if v >= 0 else v - 0.3
        ha = 'left' if v >= 0 else 'right'
        ax_bar.text(x_pos, bar.get_y() + bar.get_height() / 2,
                    f'{v:.1f}', va='center', ha=ha, fontsize=7, fontweight='bold')

ax_bar.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
xlim = ax_bar.get_xlim()
ax_bar.axvspan(0, max(xlim[1], 5), alpha=0.04, color='green')
ax_bar.axvspan(min(xlim[0], -5), 0, alpha=0.04, color='red')
ax_bar.text(max(xlim[1], 5) * 0.7, n_p - 0.1, 'NIRS benefit', fontsize=7,
            color='green', alpha=0.6, ha='center', style='italic')
ax_bar.text(min(xlim[0], -5) * 0.7, n_p - 0.1, 'IMV benefit', fontsize=7,
            color='red', alpha=0.6, ha='center', style='italic')

ylabels = []
for pname, pdata, pcolor, pstyle, plabel in patients_list:
    tx = 'NIRS' if pdata['Treatment_W'] == 1 else 'IMV'
    out = 'Survived' if pdata['delta'] == 1 else 'Died'
    vfd = pdata['vfd28']
    ylabels.append(f'{pname}\n({plabel})\n{tx}, {out}, VFD={vfd:.0f}')

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(ylabels, fontsize=8)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Estimated ITE (VFD-28 days)', fontsize=10)
ax_bar.set_title('(B) ITE Estimates by Method', fontsize=11, fontweight='bold')
ax_bar.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15),
              ncol=3, fontsize=8, frameon=True)

ax_radar.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15),
                ncol=2, fontsize=8, frameon=True)

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig(f'{OUTPUT_DIR}/fig_patient_radar_ite.png', dpi=300, bbox_inches='tight')
plt.show()
